# שלב 3 — שיטה C: Clustering של מסלולים עם MinHash-LSH

שיטה מבוססת **Clustering** (דרישת חובה). הרעיון:

1. כל מסלול = **קבוצת תאי H3** שהוא חוצה.
2. **MinHash-LSH** (מבנה מקורב מבוסס-Hash) מוצא ביעילות זוגות מסלולים דומים (דמיון Jaccard) בלי להשוות את כל הזוגות.
3. **שכנות מקומית**: לכל מסלול-זרע, האשכול = הזרע + שכניו הדומים. (לא רכיבים קשירים — כדי להימנע ממיזוג טרנזיטיבי לאשכול ענק אחד.)
4. **קונצנזוס**: בכל אשכול מחלצים את תת-המסלול הרציף הארוך ביותר שעובר דרכו לפחות `FRAC` מחברי האשכול.
5. **אימות גלובלי**: מוודאים שכל מועמד אכן פופולרי (≥minsup נסיעות בכל הדאטה) — ומסננים למקסימליים, כמו בשאר השיטות.

> זו שיטה **מקורבת/היוריסטית** — מהירה ונותנת זווית שונה, אך לא ממצה את כל המסלולים (recall נמוך יותר מהשיטות המדויקות). זה בדיוק מה שמעניין להשוות.


## 1. SparkSession + קונפיגורציה

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import CountVectorizer, MinHashLSH
from collections import defaultdict
import math, time, json, pandas as pd
from math import radians, sin, cos, asin, sqrt
import h3

spark = (SparkSession.builder.appName("porto-taxi-stage3-clustering")
         .master("local[*]").config("spark.driver.memory","4g")
         .config("spark.sql.shuffle.partitions","16").getOrCreate())
spark.sparkContext.setLogLevel("WARN")

USE_SAMPLE = True
if USE_SAMPLE:
    INPUT_PATH  = "out/trips_h3_sample.parquet"
    OUTPUT_PATH = "out/subroutes_clustering_sample.parquet"
else:
    INPUT_PATH  = "gs://YOUR_BUCKET/encoded/trips_h3.parquet"
    OUTPUT_PATH = "gs://YOUR_BUCKET/results/subroutes_clustering.parquet"

X_PCT = 0.5
CONFIGS_KM = [1, 3, 5, 10, 20, 40]
LSH_THRESHOLD = 0.3     # מרחק Jaccard מקסימלי בין מסלולים "דומים"
NUM_HASH_TABLES = 4
FRAC = 0.5              # שבר חברי האשכול שצריכים לעבור בתת-המסלול
NEIGH_CAP = 150         # תקרת שכנים לאשכול (יעילות)


## 2. פיצ'רים + MinHash-LSH → זוגות דומים

In [ ]:
_t0 = time.time()
trips = (spark.read.parquet(INPUT_PATH).select("TRIP_ID","cells","n_cells")
         .filter("n_cells>=2").withColumn("tid", F.monotonically_increasing_id()).cache())
N = trips.count(); minsup = max(2, int(N * X_PCT / 100))
print(f"N={N}  minsup={minsup}")

feat = trips.withColumn("uc", F.array_distinct("cells"))
cv = CountVectorizer(inputCol="uc", outputCol="vec", binary=True).fit(feat)
vecs = cv.transform(feat).select("tid", "vec")

mh = MinHashLSH(inputCol="vec", outputCol="hashes", numHashTables=NUM_HASH_TABLES).fit(vecs)
pairs = (mh.approxSimilarityJoin(vecs, vecs, LSH_THRESHOLD, distCol="d")
           .select(F.col("datasetA.tid").alias("a"), F.col("datasetB.tid").alias("b"))
           .filter("a < b"))
print("similar pairs:", pairs.count())


## 3. שכנויות מקומיות + קונצנזוס לכל אשכול

In [ ]:
adj = defaultdict(list)
for e in pairs.collect():
    adj[e["a"]].append(e["b"]); adj[e["b"]].append(e["a"])
print("seeds with neighbors:", len(adj))

cells_by_tid = {r["tid"]: r["cells"] for r in trips.select("tid","cells").collect()}

def consensus(rep, members):
    size = len(members); need = max(2, math.ceil(FRAC * size))
    setlist = [set(s) for s in members]
    cnt = [sum(1 for st in setlist if rep[i] in st) for i in range(len(rep))]
    bs = be = -1; bl = 0; i = 0
    while i < len(rep):
        if cnt[i] >= need:
            j = i
            while j + 1 < len(rep) and cnt[j+1] >= need: j += 1
            if j - i + 1 > bl: bl = j - i + 1; bs, be = i, j
            i = j + 1
        else:
            i += 1
    return rep[bs:be+1] if bl >= 2 else None

candidates = set()
for seed, neigh in adj.items():
    if len(neigh) > NEIGH_CAP: neigh = neigh[:NEIGH_CAP]
    members = [cells_by_tid[seed]] + [cells_by_tid[t] for t in neigh]
    sub = consensus(cells_by_tid[seed], members)
    if sub: candidates.add(tuple(sub))
print("candidate sub-routes:", len(candidates))


## 4. אימות פופולריות גלובלי + מקסימליות

In [ ]:
def windows_of_len(df, L):
    e = (f"CASE WHEN size(cells) >= {L} THEN transform(sequence(1, size(cells)-{L}+1), i -> slice(cells,i,{L})) "
         f"ELSE cast(array() as array<array<string>>) END")
    return (df.filter(F.col("n_cells") >= L).withColumn("w", F.expr(e))
              .select("TRIP_ID", F.explode("w").alias("win"))
              .withColumn("win_key", F.array_join("win", ">"))
              .dropDuplicates(["TRIP_ID", "win_key"]))

by_len = defaultdict(set)
for c in candidates: by_len[len(c)].add(">".join(c))

verified = {}
for L, keys in by_len.items():
    supp = windows_of_len(trips, L).groupBy("win_key").count()
    for r in supp.filter(F.col("win_key").isin(list(keys))).collect():
        if r["count"] >= minsup: verified[r["win_key"]] = r["count"]
print("verified popular candidates:", len(verified))

def maximal_filter(keys):
    ks = [tuple(k.split(">")) for k in keys]; ext = set()
    for k in ks:
        if len(k) >= 3: ext.add(">".join(k[:-1])); ext.add(">".join(k[1:]))
        elif len(k) == 2: ext.add(k[0]); ext.add(k[1])
    return [">".join(k) for k in ks if ">".join(k) not in ext]

maxk = maximal_filter(verified.keys())
print("maximal popular sub-routes:", len(maxk))


## 5. אורך (ק"מ), תוצאות לפי קונפיגורציה, שמירה

In [ ]:
def route_km(key):
    pts = [h3.cell_to_latlng(c) for c in key.split(">")]; t = 0.0
    for (la1, lo1), (la2, lo2) in zip(pts, pts[1:]):
        dlat, dlon = radians(la2-la1), radians(lo2-lo1)
        a = sin(dlat/2)**2 + cos(radians(la1))*cos(radians(la2))*sin(dlon/2)**2
        t += 2 * 6371 * asin(sqrt(a))
    return float(t)

pdf = pd.DataFrame([(k, len(k.split(">")), verified[k], route_km(k)) for k in maxk],
                   columns=["win_key", "n_cells", "support", "km"])
for K in CONFIGS_KM:
    print(f"  >= {K:>2} km : {int((pdf.km >= K).sum())} popular sub-routes")
if len(pdf):
    print("\ntop 10 longest:")
    print(pdf.sort_values("km", ascending=False).head(10).to_string(index=False))

spark.createDataFrame(pdf).write.mode("overwrite").parquet(OUTPUT_PATH)
print("saved:", OUTPUT_PATH)


### שמירת מטריקות להשוואה

In [ ]:
nnz = vecs.count()
metrics = {
    "method": "clustering_lsh",
    "structure": f"MinHash-LSH (Jaccard, {NUM_HASH_TABLES} tables) + neighborhood consensus",
    "distributed": True,
    "runtime_sec": round(time.time() - _t0, 1),
    "n_maximal": len(maxk),
    "struct_mem_bytes": int(NUM_HASH_TABLES * nnz * 8),   # ~MinHash signatures
    "minsup": minsup, "X_pct": X_PCT,
    "params": {"lsh_threshold": LSH_THRESHOLD, "frac": FRAC, "neigh_cap": NEIGH_CAP},
    "per_config": {str(K): int((pdf.km >= K).sum()) for K in CONFIGS_KM},
}
with open("out/metrics_clustering.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(metrics)


## סיכום השיטה

**יתרונות:** MinHash-LSH מוצא מסלולים דומים ביעילות (לא O(N²)); נותן זווית "אשכולות תנועה" וקבוצות מסלולים אופייניות.
**חסרונות:** היוריסטי — תלוי בסף ה-LSH ובגרנולריות האשכולות; **recall נמוך** יותר מהשיטות המדויקות (מפספס מסלולים שלא צפים כקונצנזוס של אף שכנות), אך **precision סביר** (מה שהוא מוצא — באמת פופולרי, כי מאומת גלובלית).

ההשוואה המלאה מול שיטות A/B במחברת `05_comparison.ipynb`.
